# Avance 02 — INEGI ATUS — Etapa 1: delimitación
**Equipo A&C | Inteligencia de Negocios**

En esta etapa leemos los datos anuales originales de **Accidentes de Tránsito Terrestre en Zonas Urbanas y Suburbanas (ATUS)**, sin modificarlos. Delimitamos a **2018 en adelante** y a las **16 alcaldías de Ciudad de México** según el catálogo `tc_municipio.csv`.

**Unidad de observación:** registro de accidente ATUS. No unimos directamente con C5 ni SSC: sus coberturas, claves y granularidad pueden diferir.

El ZIP contiene archivos anuales de 1997 a 2025; solo procesamos 2018–2025. La presencia de 2025 no implica que todos los años tengan cobertura comparable; lo verificaremos en Etapa 2.

## 1. Localizar la carpeta de datos o el ZIP
Buscamos primero `CARPETA_DATOS.zip` y después una carpeta extraída. Si existen varias copias, configuramos la ruta manual para evitar mezclar versiones. No extraemos todos los archivos si usamos el ZIP.

**Corrección:** filtramos exclusivamente la carpeta `INEGI` porque el ZIP también contiene una copia histórica en `Kaggle` con nombres de archivo coincidentes.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import zipfile
import re
from IPython.display import display

RUTA_MANUAL = None  # Ejemplo: Path('/home/usuario/Avance02/CARPETA_DATOS.zip')
ANIO_MINIMO = 2018
ANIO_MAXIMO = 2025  # Años disponibles en el ZIP recibido; modificar si se amplía la fuente

if RUTA_MANUAL is not None:
    ORIGEN = Path(RUTA_MANUAL).expanduser().resolve()
else:
    actual = Path.cwd().resolve()
    opciones = []
    for base in [actual, *actual.parents]:
        opciones += [base / 'CARPETA_DATOS.zip', base / 'CARPETA_DATOS']
    opciones = [p for p in opciones if p.exists()]
    if not opciones:
        opciones = list(Path.home().rglob('CARPETA_DATOS.zip')) + list(Path.home().rglob('CARPETA_DATOS'))
    opciones = sorted(set(p.resolve() for p in opciones))
    if len(opciones) != 1:
        print('Candidatas:', *opciones, sep='\n - ')
        raise RuntimeError('Indica una ruta única en RUTA_MANUAL (ZIP o carpeta extraída).')
    ORIGEN = opciones[0]

ES_ZIP = ORIGEN.is_file() and ORIGEN.suffix.lower() == '.zip'
if not ES_ZIP and not ORIGEN.is_dir():
    raise ValueError('La ruta debe apuntar al ZIP o a una carpeta extraída.')
zip_obj = zipfile.ZipFile(ORIGEN) if ES_ZIP else None
rutas = zip_obj.namelist() if ES_ZIP else [str(p.relative_to(ORIGEN)) for p in ORIGEN.rglob('*') if p.is_file()]
# Solo utilizamos archivos de la fuente INEGI, nunca los de Kaggle.
rutas_inegi = [r for r in rutas if '/INEGI/' in ('/' + r.replace('\\', '/'))]
archivos = {}
for r in rutas_inegi:
    m = re.search(r'atus_anual_(\d{4})\.csv$', r, re.IGNORECASE)
    if m:
        y = int(m.group(1))
        if ANIO_MINIMO <= y <= ANIO_MAXIMO:
            archivos[y] = r
if not archivos:
    raise FileNotFoundError('No se encontraron CSV anuales de ATUS desde 2018.')
cat_rutas = [r for r in rutas_inegi if r.lower().endswith('/catalogos/tc_municipio.csv')]
if len(cat_rutas) != 1:
    raise FileNotFoundError(f'Se requiere un catálogo INEGI único; encontrados: {cat_rutas}')

def abrir(ruta):
    return zip_obj.open(ruta) if ES_ZIP else open(ORIGEN / ruta, 'rb')

SALIDA = (ORIGEN.parent if ES_ZIP else ORIGEN.parent) / 'resultados' / 'resultados_INEGI_ATUS' / 'etapa1'
SALIDA.mkdir(parents=True, exist_ok=True)
print('Origen:', ORIGEN)
print('Años:', sorted(archivos))
print('Salida:', SALIDA)
print('Catálogo INEGI:', cat_rutas[0])


Origen: /home/ed0680e4-5639-48fc-ba69-7b1ab876af2f/Avance02_Proyecto/Proyecto Inteligencia de Negocios/CARPETA_DATOS.zip
Años: [2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]
Salida: /home/ed0680e4-5639-48fc-ba69-7b1ab876af2f/Avance02_Proyecto/Proyecto Inteligencia de Negocios/resultados/resultados_INEGI_ATUS/etapa1
Catálogo INEGI: CARPETA_DATOS/INEGI/conjunto_de_datos_atus_anual_csv/catalogos/tc_municipio.csv


## 2. Validar las claves oficiales de las alcaldías
Leemos `tc_municipio.csv` como texto para preservar ceros iniciales. El catálogo de CDMX contiene 16 claves de alcaldía y una categoría adicional `09999` (*Otros municipios*), que no trataremos como alcaldía.

In [2]:
with abrir(cat_rutas[0]) as f:
    catalogo = pd.read_csv(f, dtype=str, encoding='utf-8-sig')
cdmx = catalogo.loc[(catalogo['CVE_ENT'] == '09') & (catalogo['CVE_MUN'] != '999')].copy()
claves = set(cdmx['CVEGEO'].str.zfill(5))
assert len(claves) == 16, f'Esperábamos 16 alcaldías; obtuvimos {len(claves)}'
mapa_alcaldias = dict(zip(cdmx['CVEGEO'].str.zfill(5), cdmx['NOM_MUNICIPIO']))
display(cdmx[['CVEGEO','CVE_MUN','NOM_MUNICIPIO']].sort_values('CVEGEO'))
cdmx.to_csv(SALIDA / 'catalogo_alcaldias_CDMX_INEGI.csv', index=False, encoding='utf-8-sig')

,CVEGEO,CVE_MUN,NOM_MUNICIPIO
283,09002,002,Azcapotzalco
284,09003,003,Coyoacán
285,09004,004,Cuajimalpa de Morelos
286,09005,005,Gustavo A. Madero
287,09006,006,Iztacalco
288,09007,007,Iztapalapa
289,09008,008,La Magdalena Contreras
290,09009,009,Milpa Alta
291,09010,010,Álvaro Obregón
292,09011,011,Tláhuac


## 3. Reglas de delimitación
Conservamos únicamente filas que cumplen **todas** estas condiciones:

- `ANIO` entre 2018 y 2025.
- `CVE_ENT` igual a `09`.
- `CVEGEO` corresponde a una de las 16 claves oficiales del catálogo.
- `CVE_MUN` concuerda con los tres últimos dígitos de `CVEGEO`.

Separamos los registros excluidos y documentamos **todos los motivos que correspondan**. No modificamos columnas originales; agregamos `archivo_origen`, `fila_origen`, `alcaldia_normalizada` y `motivo_exclusion`.

**Importante:** ATUS tiene indicadores de personas heridas y fallecidas, pero no los sumamos todavía; su validación corresponde a la Etapa 2.

In [3]:
INVENTARIO = []
FRECUENCIAS = []
MOTIVOS = []
FILAS_APTAS = 0
FILAS_EXCLUIDAS = 0
COLUMNAS_REFERENCIA = None
APTO = SALIDA / 'INEGI_ATUS_etapa1.csv'
EXCLUIDO = SALIDA / 'INEGI_ATUS_excluidos_etapa1.csv'
for p in [APTO, EXCLUIDO]:
    if p.exists():
        p.unlink()

for anio, ruta in sorted(archivos.items()):
    recibidas = aptas = excluidas = 0
    with abrir(ruta) as f:
        for bloque in pd.read_csv(f, dtype=str, encoding='utf-8-sig', low_memory=False, chunksize=50000):
            if COLUMNAS_REFERENCIA is None:
                COLUMNAS_REFERENCIA = bloque.columns.tolist()
            if bloque.columns.tolist() != COLUMNAS_REFERENCIA:
                raise ValueError(f'Esquema diferente en {ruta}; revisar antes de unir.')
            for c in ['CVEGEO','CVE_ENT','CVE_MUN','ANIO']:
                if c not in bloque.columns:
                    raise KeyError(f'Falta {c} en {ruta}')
            recibidas += len(bloque)
            df = bloque.copy()
            df['archivo_origen'] = Path(ruta).name
            df['fila_origen'] = np.arange(recibidas-len(df)+2, recibidas+2)
            ent = df['CVE_ENT'].fillna('').str.strip().str.zfill(2)
            geo = df['CVEGEO'].fillna('').str.strip().str.zfill(5)
            mun = df['CVE_MUN'].fillna('').str.strip().str.zfill(3)
            y = pd.to_numeric(df['ANIO'], errors='coerce')
            r_anio = y.between(ANIO_MINIMO, ANIO_MAXIMO)
            r_ent = ent.eq('09')
            r_geo = geo.isin(claves)
            r_mun = mun.eq(geo.str[-3:])
            valido = r_anio & r_ent & r_geo & r_mun
            df['alcaldia_normalizada'] = geo.map(mapa_alcaldias)
            motivos = np.select(
                [~r_anio, ~r_ent, ~r_geo, ~r_mun],
                ['anio_fuera_rango','fuera_CDMX','clave_alcaldia_no_valida','clave_municipio_inconsistente'],
                default=''
            )
            # Motivo principal para el CSV; las frecuencias de cada regla se guardan aparte.
            df['motivo_exclusion'] = motivos
            for motivo, mask in [('anio_fuera_rango',~r_anio),('fuera_CDMX',~r_ent),
                                  ('clave_alcaldia_no_valida',~r_geo),('clave_municipio_inconsistente',~r_mun)]:
                MOTIVOS.append({'archivo':Path(ruta).name, 'motivo':motivo, 'filas':int(mask.sum())})
            buenos = df.loc[valido]
            malos = df.loc[~valido]
            buenos.to_csv(APTO, mode='a', header=not APTO.exists(), index=False, encoding='utf-8-sig')
            malos.to_csv(EXCLUIDO, mode='a', header=not EXCLUIDO.exists(), index=False, encoding='utf-8-sig')
            aptas += len(buenos)
            excluidas += len(malos)
            FRECUENCIAS.append({'anio_archivo':anio, 'anio_dato':anio,
                                'filas_aptas_bloque':len(buenos)})
    INVENTARIO.append({'archivo':Path(ruta).name, 'anio_archivo':anio,
                       'filas_recibidas':recibidas, 'filas_aptas':aptas,
                       'filas_excluidas':excluidas, 'columnas_originales':len(COLUMNAS_REFERENCIA)})
    FILAS_APTAS += aptas
    FILAS_EXCLUIDAS += excluidas
    print(f'{anio}: recibidas={recibidas:,}; aptas={aptas:,}; excluidas={excluidas:,}')

inventario_df = pd.DataFrame(INVENTARIO)
resumen_df = pd.DataFrame([{'filas_recibidas':int(inventario_df['filas_recibidas'].sum()),
                            'filas_aptas':FILAS_APTAS, 'filas_excluidas':FILAS_EXCLUIDAS,
                            'archivos_procesados':len(archivos)}])
assert resumen_df.iloc[0]['filas_recibidas'] == FILAS_APTAS + FILAS_EXCLUIDAS
display(inventario_df)
display(resumen_df)
inventario_df.to_csv(SALIDA / 'inventario_crudos_INEGI_ATUS.csv',index=False,encoding='utf-8-sig')
resumen_df.to_csv(SALIDA / 'resumen_delimitacion_INEGI_ATUS.csv',index=False,encoding='utf-8-sig')
pd.DataFrame(MOTIVOS).groupby(['archivo','motivo'],as_index=False)['filas'].sum().to_csv(
    SALIDA / 'frecuencias_motivos_exclusion_INEGI_ATUS.csv',index=False,encoding='utf-8-sig')

2018: recibidas=365,281; aptas=11,656; excluidas=353,625
2019: recibidas=362,729; aptas=10,673; excluidas=352,056
2020: recibidas=301,678; aptas=6,549; excluidas=295,129
2021: recibidas=340,415; aptas=6,743; excluidas=333,672
2022: recibidas=377,231; aptas=7,596; excluidas=369,635
2023: recibidas=381,048; aptas=7,276; excluidas=373,772
2024: recibidas=374,949; aptas=6,655; excluidas=368,294
2025: recibidas=380,391; aptas=5,276; excluidas=375,115


,archivo,anio_archivo,filas_recibidas,filas_aptas,filas_excluidas,columnas_originales
0,atus_anual_2018.csv,2018,365281,11656,353625,46
1,atus_anual_2019.csv,2019,362729,10673,352056,46
2,atus_anual_2020.csv,2020,301678,6549,295129,46
3,atus_anual_2021.csv,2021,340415,6743,333672,46
4,atus_anual_2022.csv,2022,377231,7596,369635,46
5,atus_anual_2023.csv,2023,381048,7276,373772,46
6,atus_anual_2024.csv,2024,374949,6655,368294,46
7,atus_anual_2025.csv,2025,380391,5276,375115,46


,filas_recibidas,filas_aptas,filas_excluidas,archivos_procesados
0,2883722,62424,2821298,8


## 4. Frecuencias de la tabla delimitada
Volvemos a leer **solo la salida** por bloques para obtener conteos por año y alcaldía, sin mantener toda la tabla en memoria.

In [4]:
from collections import Counter
por_anio = Counter()
por_alcaldia = Counter()
filas_verificadas = 0
for b in pd.read_csv(APTO, dtype=str, encoding='utf-8-sig', chunksize=50000):
    filas_verificadas += len(b)
    por_anio.update(b['ANIO'].value_counts().to_dict())
    por_alcaldia.update(b['alcaldia_normalizada'].value_counts().to_dict())
assert filas_verificadas == FILAS_APTAS
anios_df = pd.DataFrame(sorted(por_anio.items()),columns=['anio','filas'])
alcaldias_df = pd.DataFrame(sorted(por_alcaldia.items()),columns=['alcaldia','filas'])
display(anios_df)
display(alcaldias_df)
anios_df.to_csv(SALIDA / 'frecuencias_anios_INEGI_ATUS.csv',index=False,encoding='utf-8-sig')
alcaldias_df.to_csv(SALIDA / 'frecuencias_alcaldias_INEGI_ATUS.csv',index=False,encoding='utf-8-sig')
print('Filas delimitadas verificadas:', filas_verificadas)

,anio,filas
0,2018,11656
1,2019,10673
2,2020,6549
3,2021,6743
4,2022,7596
5,2023,7276
6,2024,6655
7,2025,5276


,alcaldia,filas
0,Azcapotzalco,2772
1,Benito Juárez,4830
2,Coyoacán,4348
3,Cuajimalpa de Morelos,1392
4,Cuauhtémoc,9186
5,Gustavo A. Madero,8104
6,Iztacalco,2328
7,Iztapalapa,6881
8,La Magdalena Contreras,842
9,Miguel Hidalgo,5551


Filas delimitadas verificadas: 62424


## 5. Comprobaciones y cierre
Confirmamos que las filas recibidas se distribuyen entre aptas y excluidas, que todas las filas aptas pertenecen a una alcaldía del catálogo y que se conservaron los originales. **No eliminamos duplicados en esta etapa**.

**Para la bitácora:** documentamos la delimitación con claves oficiales, la separación de exclusiones y la generación de evidencia. En la Etapa 2 revisaremos valores faltantes, folios o ausencia de identificador, fechas, severidad, duplicados y cobertura de cada año. No equipararemos automáticamente los accidentes ATUS con los reportes de C5 o SSC.

In [5]:
comprobaciones_df = pd.DataFrame([
    {'comprobacion':'conteo_recibidas_igual_aptas_mas_excluidas',
     'cumple':int(inventario_df.filas_recibidas.sum()) == FILAS_APTAS + FILAS_EXCLUIDAS},
    {'comprobacion':'frecuencias_anuales_coinciden', 'cumple':sum(por_anio.values()) == FILAS_APTAS},
    {'comprobacion':'frecuencias_alcaldias_coinciden', 'cumple':sum(por_alcaldia.values()) == FILAS_APTAS},
    {'comprobacion':'solo_16_alcaldias', 'cumple':set(por_alcaldia).issubset(set(mapa_alcaldias.values()))},
    {'comprobacion':'originales_disponibles', 'cumple':ORIGEN.exists()},
])
display(comprobaciones_df)
comprobaciones_df.to_csv(SALIDA / 'comprobaciones_INEGI_ATUS_etapa1.csv',index=False,encoding='utf-8-sig')
assert comprobaciones_df.cumple.all()
print('Etapa 1 de INEGI ATUS terminada. Comparte inventario_df y resumen_df para revisar los resultados.')
if zip_obj is not None:
    zip_obj.close() 

,comprobacion,cumple
0,conteo_recibidas_igual_aptas_mas_excluidas,True
1,frecuencias_anuales_coinciden,True
2,frecuencias_alcaldias_coinciden,True
3,solo_16_alcaldias,True
4,originales_disponibles,True


Etapa 1 de INEGI ATUS terminada. Comparte inventario_df y resumen_df para revisar los resultados.


In [3]:
from pathlib import Path
import pandas as pd

# Localizamos la carpeta de resultados
carpeta = None

for base in [Path.cwd(), *Path.cwd().parents]:
    candidata = base / "resultados/resultados_INEGI_ATUS/etapa1"

    if candidata.exists():
        carpeta = candidata
        break

if carpeta is None:
    raise FileNotFoundError(
        "No encontramos la carpeta de resultados de INEGI."
    )

# Leemos los CSV sin modificar los originales
inventario_df = pd.read_csv(
    carpeta / "inventario_crudos_INEGI_ATUS.csv",
    encoding="utf-8-sig"
).copy()

resumen_df = pd.read_csv(
    carpeta / "resumen_delimitacion_INEGI_ATUS.csv",
    encoding="utf-8-sig"
).copy()

# Mostramos los resultados completos
print("INVENTARIO DE ARCHIVOS INEGI")
print(inventario_df.to_string(index=False))

print("\nRESUMEN DE DELIMITACIÓN INEGI")
print(resumen_df.to_string(index=False))

INVENTARIO DE ARCHIVOS INEGI
            archivo  anio_archivo  filas_recibidas  filas_aptas  filas_excluidas  columnas_originales
atus_anual_2018.csv          2018           365281        11656           353625                   46
atus_anual_2019.csv          2019           362729        10673           352056                   46
atus_anual_2020.csv          2020           301678         6549           295129                   46
atus_anual_2021.csv          2021           340415         6743           333672                   46
atus_anual_2022.csv          2022           377231         7596           369635                   46
atus_anual_2023.csv          2023           381048         7276           373772                   46
atus_anual_2024.csv          2024           374949         6655           368294                   46
atus_anual_2025.csv          2025           380391         5276           375115                   46

RESUMEN DE DELIMITACIÓN INEGI
 filas_recibidas  fila